<a href="https://colab.research.google.com/github/Horizon-A028/ITA/blob/main/s8/S8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [201]:
!pip install -q rapidfuzz
import os
import pandas as pd
from pandas import DataFrame
from pandas import Series
from pandas._libs.missing import NAType
from google.colab import files
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import unicodedata
from rapidfuzz import process, fuzz
from typing import Callable

## Descripció

En aquest sprint treballaràs un procés complet d'Exploratory Data Analysis (EDA) utilitzant Python i Pandas, simulant una situació real d'anàlisi de dades dins d'una empresa.

L'objectiu no és només obtenir resultats, sinó entendre el dataset, detectar problemes, prendre decisions analítiques i justificar-les, seguint una metodologia estructurada similar a la que utilitzaria un/a analista de dades en un entorn professional.

El sprint està organitzat en tres nivells de dificultat, on cada nivell aprofundeix més en:

- Qualitat de les dades.
- Capacitat de diagnòstic.
- Tractament d'errors.
- Enriquiment del dataset.

Procura que el teu codi sigui clar i organitzat.

**Important sobre la prova de nivell**

Durant la prova:

- **No es permetrà cap connexió a internet.**
    
    Això garanteix que el teu procés de pensament sigui genuí i que puguis resoldre problemes basant-te en la lògica i els coneixements apresos.
    
- **S'avaluarà especialment el pensament lògic previ al codi.**
    
    Abans d'escriure cap línia de Python, hauràs d'explicar en paraules l'algoritme que seguiràs, pas a pas. Aquest raonament és imprescindible i forma part de la nota.
    
- **La modularitat serà clau.**
    
    Les funcions hauran d'estar ben separades, ser clares i tenir una única responsabilitat. La funció principal haurà d'orquestrar tot el procés.
    
- **Final de la prova.**
    
    Independentment de si el teu codi funciona perfectament o encara té errors, al final de la prova es permetrà la connexió a internet i se't demanarà que utilitzis una eina d'IA per millorar la teva pròpia solució. L'objectiu és valorar la teva capacitat per revisar, optimitzar i refinar el codi amb suport d'IA, tal com faries en un entorn professional.

## Nivell 1 - EDA estructurat i neteja bàsica

Tenim un dataset d'empleats obtingut a partir d'una enquesta interna. Abans de poder extreure conclusions, cal assegurar-nos que les dades són correctes, consistents i analitzables.

### Exercici 1: Exploració inicial del dataset

> 🎯 **Objectiu:** entendre l’estructura general del dataset i detectar possibles problemes abans de començar a netejar o transformar dades.

En aquest exercici has de realitzar una exploració inicial del DataFrame (sprint8_complex.xlsx) per respondre, com a mínim, a les següents preguntes:

- Quantes files i columnes té el dataset?
- Quin tipus de dades conté cada columna?
- Hi ha files duplicades?
- Les dades semblen coherents a primera vista?
- Detectes possibles problemes o inconsistències evidents?

Per fer-ho, hauràs de:

- Analitzar les dimensions del dataset
- Mostrar una mostra representativa de files
- Revisar els tipus de variables
- Les dades semblen coherents a primera vista?
- Detectar files duplicades
- Identificar i descriure possibles problemes inicials (sense solucionar-los encara)

**Important**

acompanya el codi amb un breu comentari explicant què has observat i quins problemes creus que caldrà abordar en els exercicis següents.

#### Initial Load

In [151]:
# Excercise filename
SPRINT8_COMPLEX = "sprint8_complex.xlsx"
# Upload if it doesn't exist
if not os.path.exists(SPRINT8_COMPLEX):
  uploaded = files.upload()
  name = list(uploaded.keys())[0]
  if name != SPRINT8_COMPLEX:
    os.rename(name, SPRINT8_COMPLEX)
# Make pandas read it
s800 = pd.read_excel(SPRINT8_COMPLEX)
# It is resumed by default in colab, so I don't bother to `.head()`
# Colab also allows to convert the view into an interactive table.
display(s800)

,Dades de l'enquesta de treballadors 2025,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,Generades aleatòriament per seguretat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
3,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
4,1,Marc,MuÃ±oz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1005,1002,Alba,MuÃ±oz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1006,1003,Claudia,Gil Ferrer,55297384F,Espanya,ValÃ¨ncia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
1007,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1008,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Correct Header Load

In [152]:
# Read the uploaded file again:
# Make the 4th row (index 3) the header.
s801 = pd.read_excel(SPRINT8_COMPLEX, header=3)
s801.rename(columns={s801.columns[0]: 'ID'}, inplace=True)
display(s801)

,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,1,Marc,MuÃ±oz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,TÃ¨cnic IT,4.0,NaN,NaN,NaN,16.7
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,1002,Alba,MuÃ±oz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1003,1003,Claudia,Gil Ferrer,55297384F,Espanya,ValÃ¨ncia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1005,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Duplicate Checks

In [153]:
# Helper function
def display_duplicates(df: DataFrame, subset: list[str]):
  df = df[df.duplicated(subset=subset, keep=False)]
  print(
    f"{df.drop_duplicates(subset=subset).shape[0]} duplicates "
    f"({df.shape[0]} rows) by {subset}"
  )
  df = df.sort_values(by=subset)
  display(df)

display_duplicates(s801, ["ID"])

0 duplicates (0 rows) by ['ID']


,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat


In [154]:
display_duplicates(s801, list(s801.columns[1:]))

4 duplicates (8 rows) by ['Nom', 'Cognoms', 'DNI', "País d'origen", 'Ciutat', 'Dia de Naixement', 'Mes de Naixement', 'Any de Naixement', 'Gènere', 'Salari mensual', 'Fills', 'No Fills', 'Grup Professional', 'CÃ\xa0rrec', 'Nombre_fills', 'Te_cotxe', 'Km_anuals', 'Consum_mitja_L_100km', 'Temperatura_mitjana_ciutat']


,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
372,372,Alba,MuÃ±oz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1002,1002,Alba,MuÃ±oz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
730,730,Claudia,Gil Ferrer,55297384F,Espanya,ValÃ¨ncia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
1003,1003,Claudia,Gil Ferrer,55297384F,Espanya,ValÃ¨ncia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
36,464,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.0,9.0,1992.0,D,"1,614 â‚¬",True,NaN,Grup B,Analista,3.0,NaN,NaN,NaN,10.6
40,1001,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.0,9.0,1992.0,D,"1,614 â‚¬",True,NaN,Grup B,Analista,3.0,NaN,NaN,NaN,10.6
757,757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9


In [155]:
display_duplicates(s801, ["DNI"])

9 duplicates (24 rows) by ['DNI']


,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
604,604,Laia,Navarro,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,"1,676 â‚¬",True,False,Grup A,Analista,1.0,True,50.0,NaN,14.2
250,250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,1235 euros,True,True,Grup D,Analista junior,0.0,False,NaN,NaN,10.3
759,759,Oriol,GÃ³mez,16618423V,Espanya,Sevilla,10.0,1.0,1968.0,NaN,"2,000 â‚¬",NaN,NaN,Grup A,Responsable de vendes,5.0,NaN,NaN,NaN,20.7
758,758,Oriol,GÃ³mez,16618423V,Espanya,MÃ laga,10.0,1.0,1968.0,H,1879 euros,True,NaN,Grup B,Data Analyst,4.0,False,NaN,NaN,NaN
754,754,Oriol,GÃ³mez,16618423V,Espanya,Barcelona,10.0,1.0,1968.0,Home,2.358 â‚¬,True,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
654,654,Oriol,GÃ³mez,16618423V,Espanya,Madrid,10.0,1.0,1968.0,Home,1731 euros,True,NaN,Grup B,Data analyst,4.0,NaN,NaN,NaN,NaN
692,692,Alexia,PÃ©rez DÃ­az,49529438G,Espanya,MÃ laga,12.0,10.0,1981.0,Dona,1663â‚¬,NaN,True,Grup C,Cap de projecte,NaN,True,80000.0,10.7,16.9
558,558,Alexia,PÃ©rez DÃ­az,49529438G,Espanya,MÃ laga,12.0,10.0,1981.0,Dona,"1,176 â‚¬",NaN,True,Grup A,Data Analyst,0.0,False,NaN,NaN,38.0
757,757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9


#### Type Checking

In [156]:
def extract_types(df: DataFrame) -> DataFrame:
  py_types = {}
  for col in df.columns:
    unique = {type(val).__name__ for val in df[col]}
    types = ", ".join(list(unique))
    py_types[col] = types
  types_df = pd.DataFrame(df.dtypes, columns=['pandas'])
  types_df['python'] = types_df.index.map(py_types)
  return types_df

ts801 = extract_types(s801)
display(ts801)

,pandas,python
ID,int64,int
Nom,object,"float, str"
Cognoms,object,"float, str"
DNI,object,"float, str"
País d'origen,object,"float, str"
Ciutat,object,"float, str"
Dia de Naixement,float64,float
Mes de Naixement,float64,float
Any de Naixement,float64,float
Gènere,object,"float, str"


#### DataFrame Shape

In [157]:
r, c = s801.shape
print(f"The dataset has {r} rows and {c} columns.")

The dataset has 1007 rows and 20 columns.


#### Problems

- The dates are stored as multiple float values instead of in a single date fromat.
- Salay should be a number + the monetary symbol in a column apart.
- Mangled text due to incorrect file metadata making pandas use the wrong decode function.
- All sorts of duplicates.
- Non-normalized values.
- ...

### Exercici 2: Validacions bàsiques de les dades i decisions d'eliminació

> 🎯 **Objectiu:** assegurar que les dades compleixen unes condicions mínimes de qualitat i determinar quines files poden continuar sent analitzades i quines s'han de descartar.

En aquest exercici hauràs de validar aspectes clau del dataset que són crítics en qualsevol context real d'anàlisi de dades, prenent decisions justificades sobre la **validesa o eliminació de registres**.

Aspectes a validar:

- Existeixen **identificadors únics (DNI)** per a cada persona?
- Què faràs amb les **files duplicades** o registres repetits?
- Hi ha **valors obligatoris** que falten o informació incoherent?
- Detectes **formats incorrectes**, errors de codificació o valors impossibles?

Concretament, hauràs de:

- Corregir possibles problemes de codificació i accentuació en els noms de les columnes i en els valors de text, si escau.
- Detectar i eliminar files duplicades exactes, indicant quants registres s'han eliminat.
- Comprovar la unicitat del DNI. En cas de detectar DNI duplicats, hauràs d'analitzar la coherència de la informació associada i descartar o anul·lar els registres que no siguin fiables, justificant clarament la decisió presa.
- Identificar els camps clau que no haurien de contenir valors nuls, detectar registres amb informació insuficient i eliminar únicament aquelles files completament buides o amb informació irrecuperable o no substituïble, explicant per què no és possible ni raonable imputar-les.

**Important**

En aquest exercici no s'espera que imputis dades, sinó que **decideixis quins registres poden formar part de l'anàlisi i quins no**, argumentant sempre el criteri utilitzat.

🧠 Si el text "s'entén, però està mal escrit" → codificació

Si el text "està ben escrit, però no és útil per a l'anàlisi" → validació

#### Fix Encoding

```
excel : [utf-8] -> ... -> [utf-8] labeled as cp1252
pandas: [utf-8] -cp1252-decode-> [python-str] mangled
fix   : [python-str] -cp1252-encode-> [utf-8] original
        [utf-8] -utf-8-decode-> [python-str] correct
```

In [158]:
def fix_encoding(df: DataFrame) -> DataFrame:
  def enc_dec(s):
    if not isinstance(s, str): return s
    try: return s\
      .encode("cp1252")\
      .decode("utf-8")
    except Exception: return s
  df.columns = df.columns.map(enc_dec)
  for col in df.columns:
    df[col] = df[col].apply(enc_dec)
  return df

s802 = fix_encoding(s801)
display(s802)

,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 €",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 €",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,Tècnic IT,4.0,NaN,NaN,NaN,16.7
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 €",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 €",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,1002,Alba,Muñoz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 €",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1003,1003,Claudia,Gil Ferrer,55297384F,Espanya,València,1.0,11.0,1981.0,H,1.789 €,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786€,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1005,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Process DNI Duplicates

In [159]:
display_duplicates(s802, ["DNI"])
s803 = s802.drop_duplicates(subset=s802.columns[1:], keep="first")
display_duplicates(s803, ["DNI"])

9 duplicates (24 rows) by ['DNI']


,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
604,604,Laia,Navarro,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,"1,676 €",True,False,Grup A,Analista,1.0,True,50.0,NaN,14.2
250,250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,1235 euros,True,True,Grup D,Analista junior,0.0,False,NaN,NaN,10.3
759,759,Oriol,Gómez,16618423V,Espanya,Sevilla,10.0,1.0,1968.0,NaN,"2,000 €",NaN,NaN,Grup A,Responsable de vendes,5.0,NaN,NaN,NaN,20.7
758,758,Oriol,Gómez,16618423V,Espanya,Màlaga,10.0,1.0,1968.0,H,1879 euros,True,NaN,Grup B,Data Analyst,4.0,False,NaN,NaN,NaN
754,754,Oriol,Gómez,16618423V,Espanya,Barcelona,10.0,1.0,1968.0,Home,2.358 €,True,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
654,654,Oriol,Gómez,16618423V,Espanya,Madrid,10.0,1.0,1968.0,Home,1731 euros,True,NaN,Grup B,Data analyst,4.0,NaN,NaN,NaN,NaN
692,692,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,1663€,NaN,True,Grup C,Cap de projecte,NaN,True,80000.0,10.7,16.9
558,558,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,"1,176 €",NaN,True,Grup A,Data Analyst,0.0,False,NaN,NaN,38.0
757,757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786€,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786€,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9


5 duplicates (16 rows) by ['DNI']


,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
250,250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,1235 euros,True,True,Grup D,Analista junior,0.0,False,NaN,NaN,10.3
604,604,Laia,Navarro,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,"1,676 €",True,False,Grup A,Analista,1.0,True,50.0,NaN,14.2
654,654,Oriol,Gómez,16618423V,Espanya,Madrid,10.0,1.0,1968.0,Home,1731 euros,True,NaN,Grup B,Data analyst,4.0,NaN,NaN,NaN,NaN
754,754,Oriol,Gómez,16618423V,Espanya,Barcelona,10.0,1.0,1968.0,Home,2.358 €,True,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
758,758,Oriol,Gómez,16618423V,Espanya,Màlaga,10.0,1.0,1968.0,H,1879 euros,True,NaN,Grup B,Data Analyst,4.0,False,NaN,NaN,NaN
759,759,Oriol,Gómez,16618423V,Espanya,Sevilla,10.0,1.0,1968.0,NaN,"2,000 €",NaN,NaN,Grup A,Responsable de vendes,5.0,NaN,NaN,NaN,20.7
558,558,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,"1,176 €",NaN,True,Grup A,Data Analyst,0.0,False,NaN,NaN,38.0
692,692,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,1663€,NaN,True,Grup C,Cap de projecte,NaN,True,80000.0,10.7,16.9
89,89,Guillem,Pujol Costa,54013571W,Espanya,A Coruña,17.0,7.0,1983.0,D,1.574 €,True,NaN,Grup B,Project lead,4.0,False,NaN,NaN,14.6
142,142,Marta,Ruiz,54013571W,Espanya,Palma,24.0,5.0,1982.0,H,"1,276 €",True,True,Grup A,Analista,0.0,False,NaN,NaN,15.7


In [160]:
# If this is a record database, then each case has to be resolved
# through consultation. If this data is to be analyzed, we can either
# keep one record of each person, or delete them to keep the data
# clean. In this case I will assume that the duplicates are chronological
# entries, where the latest is the source of truth. In cases where
# multiple people share the same DNI, I will invalidate the DNI
# entry since it is unreliable, but keep the person's data.

# Since there are few cases, each with their own peculiarities,
# I will treat them manually.

s804 = s803.drop([
  25,
  913, # Sara has three quasy identical entries, leave the last
  250  # Laina has upgraded to Analyst from Junior
])

# Since we are dealing with data analytics, it is reasonable that
# We don't want to account for one person more than one time, thus
# why I deleted some rows.

# Add Laina's missing surname "Vila"
s804.loc[s804["ID"] == 604, "Cognoms"] = "Navarro Vila"

# Multiple people sharing DNI, the DNI is invalid:
# - Better a NaN than a confusing invalid value.
# - If I can, I'd alert the company of these instances.
s804.loc[s804["DNI"] == "90585882A", "DNI"] = pd.NA
s804.loc[s804["DNI"] == "54013571W", "DNI"] = pd.NA

# Remove Alexia's old registry
# On the new registry she chose not to specify whether she has kids
# Also she has a new car.
s804 = s804.drop(558)

# Remove Oriol's old registry
# Can't change gender, or unhave kids*.
# Since the changes are not explicit and simply not filled,
# I will extend the previous values on to the new registry.
s804.loc[s804["ID"] == 759, "Gènere"] = "H"
s804.loc[s804["ID"] == 759, "Fills"] = True
s804 = s804.drop([
  654,
  754,
  758
])

display_duplicates(s804, ["DNI"])


1 duplicates (6 rows) by ['DNI']


,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
89,89,Guillem,Pujol Costa,<NA>,Espanya,A Coruña,17.0,7.0,1983.0,D,1.574 €,True,NaN,Grup B,Project lead,4.0,False,NaN,NaN,14.6
104,104,Oriol,Vila Casals,<NA>,Espanya,Madrid,30.0,9.0,1964.0,H,1987 euros,NaN,True,Grup B,Responsable de vendes,0.0,False,NaN,NaN,13.5
114,114,Pol,Serra,<NA>,Espanya,Girona,4.0,12.0,1960.0,H,"1,178 €",NaN,True,Grup A,Data Analyst,NaN,False,NaN,NaN,16.2
142,142,Marta,Ruiz,<NA>,Espanya,Palma,24.0,5.0,1982.0,H,"1,276 €",True,True,Grup A,Analista,0.0,False,NaN,NaN,15.7
281,281,Xavier,Vidal,<NA>,Espanya,Alacant,2.0,7.0,1958.0,H,1179 euros,NaN,NaN,Grup A,Tècnic IT,0.0,True,63574.0,6.0,16.8
1000,1000,Sara,Serra,<NA>,Espanya,Tarragona,28.0,5.0,2002.0,D,"1,073 €",NaN,True,Grup A,Administratiu,0.0,False,NaN,NaN,15.8


#### Further Process DNI

In [161]:
def is_dni_valid(row: Series, checksum=False) -> bool:
  """
Validates a Spanish DNI based on the official checksum letter algorithm.
Returns `True` if valid, `False` if invalid or value missing.
  """
  dni = row["DNI"]
  if pd.isna(dni) or not isinstance(dni, str):
    return False
  dni: str = dni.strip().upper()
  if len(dni) != 9:
    return False
  numbers: str = dni[:-1]
  letter : str = dni[-1]
  if not numbers.isdigit() or not letter.isalpha():
    return False
  if not checksum:
    return True
  table = "TRWAGMYFPDXBNJZSQVHLCKE"
  try:
    return table[int(numbers) % 23] == letter
  except ValueError:
    return False

s804["IS_DNI_VALID"] = s804.apply(
  lambda x: is_dni_valid(x, True),
  axis=1
)
# For some reason this test database didn't have into
# account the the official checksum algorithm.
display(s804[~s804["IS_DNI_VALID"]])

,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,...,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,IS_DNI_VALID
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,...,True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1,False
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,...,NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7,False
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,...,True,NaN,Grup A,Tècnic IT,4.0,NaN,NaN,NaN,16.7,False
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,...,True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3,False
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,...,NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999,999,Adrià,Vila Díaz,57511543T,Espanya,Palma,31.0,12.0,1991.0,H,...,True,NaN,Grup C,Analista,1.0,False,NaN,NaN,-10.0,False
1000,1000,Sara,Serra,<NA>,Espanya,Tarragona,28.0,5.0,2002.0,D,...,NaN,True,Grup A,Administratiu,0.0,False,NaN,NaN,15.8,False
1001,651,Kasper,Pettersen,62042563B,Noruega,Oslo,11.0,5.0,1996.0,D,...,NaN,True,Grup B,Senior analyst,0.0,True,80000.0,0.5,6.5,False
1005,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


In [162]:
s804["IS_DNI_VALID"] = s804.apply(is_dni_valid, axis=1)
display(s804[~s804["IS_DNI_VALID"]])

,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,...,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,IS_DNI_VALID
89,89,Guillem,Pujol Costa,<NA>,Espanya,A Coruña,17.0,7.0,1983.0,D,...,True,NaN,Grup B,Project lead,4.0,False,NaN,NaN,14.6,False
104,104,Oriol,Vila Casals,<NA>,Espanya,Madrid,30.0,9.0,1964.0,H,...,NaN,True,Grup B,Responsable de vendes,0.0,False,NaN,NaN,13.5,False
114,114,Pol,Serra,<NA>,Espanya,Girona,4.0,12.0,1960.0,H,...,NaN,True,Grup A,Data Analyst,NaN,False,NaN,NaN,16.2,False
142,142,Marta,Ruiz,<NA>,Espanya,Palma,24.0,5.0,1982.0,H,...,True,True,Grup A,Analista,0.0,False,NaN,NaN,15.7,False
281,281,Xavier,Vidal,<NA>,Espanya,Alacant,2.0,7.0,1958.0,H,...,NaN,NaN,Grup A,Tècnic IT,0.0,True,63574.0,6.0,16.8,False
1000,1000,Sara,Serra,<NA>,Espanya,Tarragona,28.0,5.0,2002.0,D,...,NaN,True,Grup A,Administratiu,0.0,False,NaN,NaN,15.8,False
1005,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1006,1006,???,###,XXXX,Narnia,Desconeguda,-99.0,99.0,3000.0,??,...,NaN,NaN,Z,???,NaN,NaN,NaN,NaN,NaN,False


In [163]:
# Delete the trolls
s805 = s804.drop([1005, 1006])
display(s805[~s805["IS_DNI_VALID"]])
# Clean up
s805 = s805.drop(columns=["IS_DNI_VALID"])

,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,...,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,IS_DNI_VALID
89,89,Guillem,Pujol Costa,<NA>,Espanya,A Coruña,17.0,7.0,1983.0,D,...,True,NaN,Grup B,Project lead,4.0,False,NaN,NaN,14.6,False
104,104,Oriol,Vila Casals,<NA>,Espanya,Madrid,30.0,9.0,1964.0,H,...,NaN,True,Grup B,Responsable de vendes,0.0,False,NaN,NaN,13.5,False
114,114,Pol,Serra,<NA>,Espanya,Girona,4.0,12.0,1960.0,H,...,NaN,True,Grup A,Data Analyst,NaN,False,NaN,NaN,16.2,False
142,142,Marta,Ruiz,<NA>,Espanya,Palma,24.0,5.0,1982.0,H,...,True,True,Grup A,Analista,0.0,False,NaN,NaN,15.7,False
281,281,Xavier,Vidal,<NA>,Espanya,Alacant,2.0,7.0,1958.0,H,...,NaN,NaN,Grup A,Tècnic IT,0.0,True,63574.0,6.0,16.8,False
1000,1000,Sara,Serra,<NA>,Espanya,Tarragona,28.0,5.0,2002.0,D,...,NaN,True,Grup A,Administratiu,0.0,False,NaN,NaN,15.8,False


#### Count NA

In [164]:
def standardize_NA(df: DataFrame) -> DataFrame:
  def value(v):
    if pd.isna(v):
      return pd.NA
    if isinstance(v, str):
      v = v.strip()
      if not v:
        return pd.NA
    return v
  return df.map(value)

s806 = standardize_NA(s805)
display(s806)

,ID,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 €",True,<NA>,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 €",<NA>,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,<NA>,Grup A,Tècnic IT,4.0,<NA>,<NA>,<NA>,16.7
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 €",True,<NA>,Grup B,Data Analyst,2.0,False,<NA>,<NA>,18.3
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 €",<NA>,True,Grup B,Administratiu,<NA>,True,<NA>,11.8,13.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,12.0,10.0,1961.0,Dona,"1,074 €",<NA>,<NA>,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0
998,998,Chloé,Dubois,66354268T,França,Nice,2.0,12.0,1958.0,H,"1,954 €",True,<NA>,Grup B,Data Analyst,1.0,<NA>,<NA>,<NA>,11.7
999,999,Adrià,Vila Díaz,57511543T,Espanya,Palma,31.0,12.0,1991.0,H,"2,010 €",True,<NA>,Grup C,Analista,1.0,False,<NA>,<NA>,-10.0
1000,1000,Sara,Serra,<NA>,Espanya,Tarragona,28.0,5.0,2002.0,D,"1,073 €",<NA>,True,Grup A,Administratiu,0.0,False,<NA>,<NA>,15.8


In [165]:
display(s806.isna().sum())

,0
ID,0
Nom,0
Cognoms,0
DNI,6
País d'origen,0
Ciutat,0
Dia de Naixement,0
Mes de Naixement,0
Any de Naixement,0
Gènere,39


### Exercici 3: Transformacions necessàries i preparació del dataset

> 🎯 **Objectiu:** preparar el dataset perquè sigui coherent, analitzable i estructuralment correcte abans de realitzar qualsevol anàlisi descriptiva.

En aquest exercici hauràs d'aplicar transformacions bàsiques però essencials, orientades a millorar la qualitat de les dades i facilitar l'anàlisi posterior.

Aspectes a revisar i transformar:

- Els valors categòrics estan normalitzats i són consistents?
- Els valors nuls són coherents amb la informació disponible?
- Hi ha redundància de variables que pugui generar problemes analítics?
- Les variables contextuals (com temperatura, salari) estan completes i preparades per a l'anàlisi?

Concretament, hauràs de:

- Normalitzar els valors categòrics per evitar ambigüitats (per exemple, valors que representen el mateix concepte però estan escrits de manera diferent).
- Revisar la coherència dels valors nuls, assegurant que:
    - Les persones que tenen quilometratge anual i consum mitjà associats disposen de cotxe. En els casos en què consti consum mitjà però no quilometratge anual, caldrà decidir com resoldre aquesta incoherència de manera justificada.
    - Les persones que tenen un valor en el nombre de fills efectivament tenen fills.
- Unificar la informació sobre fills en una única columna, ja que mantenir diverses columnes amb informació complementària no és òptim i pot generar problemes analítics (redundància o multicolinealitat).
- Validar la temperatura mitjana. En cas que aquesta informació falti, hauràs d'imputar-la utilitzant un criteri raonable (per ciutat o per un nivell superior justificat).
- Revisar i imputar els salaris faltants, aplicant criteris analítics coherents (com ara càrrec, grup professional, ciutat, gènere o franja d'edat).

**Important**

en aquest exercici no es tracta d'eliminar registres, sinó de **transformar, normalitzar i preparar les dades** perquè siguin analitzables de manera rigorosa en els nivells següents.

#### Make ID the index

In [166]:
s806 = s806.set_index("ID")
display(s806)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
ID,,,,,,,,,,,,,,,,,,,
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 €",True,<NA>,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 €",<NA>,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,<NA>,Grup A,Tècnic IT,4.0,<NA>,<NA>,<NA>,16.7
3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 €",True,<NA>,Grup B,Data Analyst,2.0,False,<NA>,<NA>,18.3
4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 €",<NA>,True,Grup B,Administratiu,<NA>,True,<NA>,11.8,13.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,12.0,10.0,1961.0,Dona,"1,074 €",<NA>,<NA>,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0
998,Chloé,Dubois,66354268T,França,Nice,2.0,12.0,1958.0,H,"1,954 €",True,<NA>,Grup B,Data Analyst,1.0,<NA>,<NA>,<NA>,11.7
999,Adrià,Vila Díaz,57511543T,Espanya,Palma,31.0,12.0,1991.0,H,"2,010 €",True,<NA>,Grup C,Analista,1.0,False,<NA>,<NA>,-10.0


#### Process Birth Date

In [167]:
def is_birth_date_valid(row: Series):
  birth_year  = row["Any de Naixement"]
  birth_month = row["Mes de Naixement"]
  birth_day   = row["Dia de Naixement"]
  if pd.isna(birth_year) or pd.isna(birth_month) or pd.isna(birth_day):
    return False
  try:
    year_i = int(birth_year)
    month_i = int(birth_month)
    day_i = int(birth_day)
    if not (1900 <= year_i <= 2010):
      return False
    pd.Timestamp(year=year_i, month=month_i, day=day_i)
    return True
  except Exception:
    return False

s806["BIRTH_DATE_VALID"] = s806.apply(is_birth_date_valid, axis=1)
display(s806[~s806["BIRTH_DATE_VALID"]])

,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,BIRTH_DATE_VALID
ID,,,,,,,,,,,,,,,,,,,,
9,Alba,García,75181648S,Espanya,Las Palmas de Gran Canaria,32.0,0.0,1940.0,H,"1,450 €",True,<NA>,Grup A,Senior analyst,4.0,False,<NA>,<NA>,20.1,False
39,Guillem,Vila,92874006W,Espanya,Murcia,0.0,99.0,3000.0,H,dos mil,<NA>,True,Grup A,Senior analyst,0.0,<NA>,<NA>,<NA>,<NA>,False
41,Paul,Richard Laurent,43567615Z,França,Bordeaux,32.0,99.0,2019.0,Dona,"1,967 €",<NA>,True,Grup B,Cap de projecte,<NA>,False,<NA>,<NA>,15.2,False
47,Aina,Fernández,38468360C,Espanya,Zaragoza,-5.0,0.0,3000.0,Home,1939 euros,True,<NA>,Grup C,Analista,1.0,True,78064.0,<NA>,<NA>,False
53,Aina,Serra,29226889H,Espanya,València,0.0,0.0,2019.0,F,"1,866 €",<NA>,True,Grup B,Tècnic IT,<NA>,False,<NA>,<NA>,21.1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
914,Alejandro,Soler López,41786590C,Espanya,Zaragoza,32.0,99.0,2026.0,<NA>,"1,095 €",<NA>,True,Grup A,Data Analyst,0.0,<NA>,<NA>,<NA>,19.8,False
926,David,Pérez,52298598B,Espanya,Murcia,-5.0,0.0,2019.0,Dona,779 €,<NA>,True,Grup A,Analista junior,0.0,<NA>,<NA>,<NA>,19.0,False
63,Ida,Johansen Kristiansen,41407343V,Noruega,Bergen,32.0,99.0,2019.0,D,"2,759 €",<NA>,True,Grup C,Senior analyst,<NA>,False,<NA>,<NA>,9.1,False


In [168]:
# We can see that all 77 invalid dates are nonsense.
# So when we transform the dates into the aproppiate format,
# these invalid dates can be left as `pd.NA`
def cast_birth_date(row: Series) -> pd.Timestamp:
  if not row["BIRTH_DATE_VALID"]:
    return pd.NA
  try:
    return pd.Timestamp(
      year=int(row["Any de Naixement"]),
      month=int(row["Mes de Naixement"]),
      day=int(row["Dia de Naixement"])
    ).date()
  except Exception:
    raise Exception(f"Failed to cast date {row}")

s807 = s806.drop(columns=[
  "BIRTH_DATE_VALID",
  "Any de Naixement",
  "Mes de Naixement",
  "Dia de Naixement"
])
s807["Birth Date"] = s806.apply(cast_birth_date, axis=1)
display(s807)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,Birth Date
ID,,,,,,,,,,,,,,,,,
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,"1,469 €",True,<NA>,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,H,"2,718 €",<NA>,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,D,1358 euros,True,<NA>,Grup A,Tècnic IT,4.0,<NA>,<NA>,<NA>,16.7,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,H,"1,478 €",True,<NA>,Grup B,Data Analyst,2.0,False,<NA>,<NA>,18.3,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,"1,284 €",<NA>,True,Grup B,Administratiu,<NA>,True,<NA>,11.8,13.1,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Dona,"1,074 €",<NA>,<NA>,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0,1961-10-12
998,Chloé,Dubois,66354268T,França,Nice,H,"1,954 €",True,<NA>,Grup B,Data Analyst,1.0,<NA>,<NA>,<NA>,11.7,1958-12-02
999,Adrià,Vila Díaz,57511543T,Espanya,Palma,H,"2,010 €",True,<NA>,Grup C,Analista,1.0,False,<NA>,<NA>,-10.0,1991-12-31


#### Rename

In [169]:
s808 = s807.rename(columns={
  "Nom": "Name",
  "Cognoms": "Surname",
  "Gènere": "M/F",
  "Salari mensual": "Salary",
  "Fills": "Kids",
  "No Fills": "Not Kids",
  "Nombre_fills": "No. Kids",
  "Grup Professional": "Pro.G.",
  "Càrrec": "Position",
  "Te_cotxe": "Car",
  "Km_anuals": "Km/year",
  "Consum_mitja_L_100km": "L/100Km",
  "Temperatura_mitjana_ciutat": "Avg.C.ºC",
  "Ciutat": "City",
  "País d'origen": "Country"
})

display(s808)

,Name,Surname,DNI,Country,City,M/F,Salary,Kids,Not Kids,Pro.G.,Position,No. Kids,Car,Km/year,L/100Km,Avg.C.ºC,Birth Date
ID,,,,,,,,,,,,,,,,,
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,"1,469 €",True,<NA>,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,H,"2,718 €",<NA>,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,D,1358 euros,True,<NA>,Grup A,Tècnic IT,4.0,<NA>,<NA>,<NA>,16.7,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,H,"1,478 €",True,<NA>,Grup B,Data Analyst,2.0,False,<NA>,<NA>,18.3,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,"1,284 €",<NA>,True,Grup B,Administratiu,<NA>,True,<NA>,11.8,13.1,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Dona,"1,074 €",<NA>,<NA>,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0,1961-10-12
998,Chloé,Dubois,66354268T,França,Nice,H,"1,954 €",True,<NA>,Grup B,Data Analyst,1.0,<NA>,<NA>,<NA>,11.7,1958-12-02
999,Adrià,Vila Díaz,57511543T,Espanya,Palma,H,"2,010 €",True,<NA>,Grup C,Analista,1.0,False,<NA>,<NA>,-10.0,1991-12-31


#### Process Kids

In [170]:
ts808 = extract_types(s808)
display(ts808)

,pandas,python
Name,object,str
Surname,object,str
DNI,object,"NAType, str"
Country,object,str
City,object,str
M/F,object,"NAType, str"
Salary,object,str
Kids,object,"NAType, bool"
Not Kids,object,"NAType, bool"
Pro.G.,object,str


In [171]:
def consolidate_kids(df: DataFrame) -> bool | NAType:
  def helper(row: Series) -> bool:
    has_kids = row["Kids"]
    if pd.isna(has_kids): has_kids = None
    no_kids  = row["Not Kids"]
    if pd.isna(no_kids): no_kids = None
    match (has_kids, no_kids):
      case (None, None):
        return pd.NA # No data
      case (None, False):
        return True # Not Not Kids
      case (None, True):
        return False # Yes Not Kids
      case (False, None):
        return False # Not Kids
      case (False, False):
        return pd.NA # Contradictory
      case (False, True):
        return False # Not Kids and Yes Not Kids
      case (True, None):
        return True # Kids
      case (True, False):
        return True # Kids and Not Not Kids
      case (True, True):
        return pd.NA # Contradictory
      case _:
        raise Exception(f"Unexpected combination: {has_kids}, {no_kids}")
  df["Kids"] = df.apply(helper, axis=1)
  df = df.drop(columns=["Not Kids"])
  return df

s809 = consolidate_kids(s808)
display(s809)

,Name,Surname,DNI,Country,City,M/F,Salary,Kids,Pro.G.,Position,No. Kids,Car,Km/year,L/100Km,Avg.C.ºC,Birth Date
ID,,,,,,,,,,,,,,,,
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,"1,469 €",True,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,H,"2,718 €",False,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,D,1358 euros,True,Grup A,Tècnic IT,4.0,<NA>,<NA>,<NA>,16.7,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,H,"1,478 €",True,Grup B,Data Analyst,2.0,False,<NA>,<NA>,18.3,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,"1,284 €",False,Grup B,Administratiu,<NA>,True,<NA>,11.8,13.1,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Dona,"1,074 €",<NA>,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0,1961-10-12
998,Chloé,Dubois,66354268T,França,Nice,H,"1,954 €",True,Grup B,Data Analyst,1.0,<NA>,<NA>,<NA>,11.7,1958-12-02
999,Adrià,Vila Díaz,57511543T,Espanya,Palma,H,"2,010 €",True,Grup C,Analista,1.0,False,<NA>,<NA>,-10.0,1991-12-31


In [172]:
class Kids:
  def __init__(self, kids: bool, num: int):
    self.kids = kids
    self.num  = num
    if not kids and num > 0:
      raise Exception(f"Invalid combination: {kids}, {num}")
    if kids and num == 0:
      raise Exception(f"Invalid combination: {kids}, {num}")
    if num < -1:
      raise Exception(f"Invalid number: {num}")
  def __str__(self):
    if self.kids:
      if self.num == -1:
        return f"{True}"
      return f"{self.num}"
    else:
      return f"{False}"
  def __repr__(self):
    return f"{self.__str__()}"
  def get_num(self) -> int | NAType:
    if self.kids:
      if self.num == -1:
        return pd.NA
      return self.num
    else:
      return pd.NA
  def get_kids(self) -> bool:
    return self.kids
  def __eq__(self, other):
    if not isinstance(other, Kids):
      return False
    return self.kids == other.kids and self.num == other.num
  def __lt__(self, other):
    if not isinstance(other, Kids):
      return False
    if self.kids != other.kids:
      return self.kids < other.kids
    return self.num < other.num
  def __hash__(self) -> int:
    return hash((self.kids, self.num))

def consolidate_kids2(df: DataFrame) -> DataFrame:
  def helper(row: Series) -> Kids | NAType:
    no_kids = row["No. Kids"]
    if pd.isna(no_kids): no_kids = None
    elif no_kids < 0   : no_kids = None
    kids = row["Kids"]
    if pd.isna(kids): kids = None
    match (kids, no_kids):
      case (None, None):
        return pd.NA # No data
      case (None, 0):
        return Kids(False, 0) # No Kids
      case (None, _):
        return Kids(True, no_kids) # Non-zero kids
      case (False, None):
        return Kids(False, 0) # No Kids
      case (False, 0):
        return Kids(False, 0) # No Kids
      case (False, _):
        return Kids(True, no_kids) # Non-zero kids, despite boolean
      case (True, None):
        return Kids(True, -1) # Unknown no. kids
      case (True, 0):
        return Kids(True, -1) # Unknonw no. kids
      case (True, _):
        return Kids(True, no_kids) # Kids
      case _:
        raise Exception(f"Unexpected combination: {kids}, {no_kids}")
  df = df.copy()
  df["Kids"] = df.apply(helper, axis=1).copy()
  df = df.drop(columns=["No. Kids"])
  return df

s810 = consolidate_kids2(s809)
display(s810)

,Name,Surname,DNI,Country,City,M/F,Salary,Kids,Pro.G.,Position,Car,Km/year,L/100Km,Avg.C.ºC,Birth Date
ID,,,,,,,,,,,,,,,
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,"1,469 €",3.0,Grup B,Cap de projecte,True,32108.0,25.0,10.1,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,H,"2,718 €",False,Grup C,Senior analyst,True,19496.0,10.4,18.7,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,D,1358 euros,4.0,Grup A,Tècnic IT,<NA>,<NA>,<NA>,16.7,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,H,"1,478 €",2.0,Grup B,Data Analyst,False,<NA>,<NA>,18.3,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,"1,284 €",False,Grup B,Administratiu,True,<NA>,11.8,13.1,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Dona,"1,074 €",1.0,Grup A,Data Analyst,True,80000.0,12.0,12.0,1961-10-12
998,Chloé,Dubois,66354268T,França,Nice,H,"1,954 €",1.0,Grup B,Data Analyst,<NA>,<NA>,<NA>,11.7,1958-12-02
999,Adrià,Vila Díaz,57511543T,Espanya,Palma,H,"2,010 €",1.0,Grup C,Analista,False,<NA>,<NA>,-10.0,1991-12-31


In [173]:
display(s810["Kids"].value_counts().sort_index())

,count
Kids,
False,569
1.0,171
2.0,79
3.0,64
4.0,78
5.0,1


#### Process Place

In [174]:
def process_place(df: DataFrame) -> DataFrame:
  GEO_FILE: str = "geo_file.csv"
  # Initialize the Nominatim Geocoder
  # Use a rate limiter to respect OpenStreetMap's usage policy (1 req/second)
  geocode = RateLimiter(
    Nominatim(user_agent="colab_city_country_validator").geocode,
    min_delay_seconds=2,
    max_retries=3
  )
  def gen_query(row: Series):
    city    = row["City"]
    country = row["Country"]
    match (city, country):
      case (None, None):
        return pd.NA
      case (None, _):
        return country
      case (_, None):
        return city
      case (_, _):
        return f"{city}, {country}"
  # Get all unique geography pairs to avoid hitting the API too many times
  def get_loc_pairs(df: DataFrame):
    df = df[["Country", "City"]].dropna(how="all").drop_duplicates()
    df["query"] = df.apply(gen_query, axis=1)
    return df
  def load_file() -> DataFrame | None:
    # Upload if it doesn't exist
    if not os.path.exists(GEO_FILE):
      uploaded = files.upload()
      if not uploaded:
        return None
      name = list(uploaded.keys())[0]
      if name != GEO_FILE:
        os.rename(name, GEO_FILE)
    # Since the file is saved with .to_csv(), read it using pd.read_csv
    return pd.read_csv(GEO_FILE)
  def craft_series(status, country, city):
    return pd.Series(
      [status, country, city],
      index=["Geo", "Country2", "City2"]
    )
  def standarize_geo(row: Series) -> Series:
    query   = row["query"]
    country = row["Country"]
    city    = row["City"]
    if pd.isna(query):
      return craft_series(pd.NA, pd.NA, pd.NA)
    try:
      location = geocode(
        query=query,
        addressdetails=True,
        language="en"
      )
      if location:
        address = location.raw.get("address", {})
        # Standardized country from the API in English
        std_country = address.get("country")
        if not std_country:
          return craft_series("404-Contry", country, city)
        # Standarized city or administrative equivalents
        std_city = \
          address.get("municipality",
          address.get("city",
          address.get("town",
          address.get("village",
          address.get("county",
          address.get("state",
          address.get("city_district")
        ))))))
        if not std_city:
          return craft_series("404-City", std_country, city)
        return craft_series("OK", std_country, std_city)
      else:
        return craft_series("404", country, city)
    except Exception as e:
      return craft_series(f"KO: {str(e)}", country, city)
  def gen_file(df: DataFrame) -> DataFrame:
    loc_pairs = get_loc_pairs(df)
    geo_info = loc_pairs.apply(standarize_geo, axis=1)
    return pd.concat([loc_pairs, geo_info], axis=1).drop(columns=["query"])

  # Save original column order
  original_columns = list(df.columns)
  geo = load_file()
  if geo is None:
    geo = gen_file(df)
    geo.to_csv(GEO_FILE, index=False)
  original_columns.insert(3, "Geo")
  return df.merge(
    geo,
    on=["Country", "City"],
    how="left"
  ).drop(columns=["Country", "City"])\
   .rename(columns={
      "Country2": "Country",
      "City2": "City"
    })[original_columns]


s811 = process_place(s810).rename(columns={
  "Birth Date": "Birth"
})
cols = list(s811.columns)[:-1]
cols.insert(2, "Birth")
s811 = s811[cols]
display(s811)

,Name,Surname,Birth,DNI,Geo,Country,City,M/F,Salary,Kids,Pro.G.,Position,Car,Km/year,L/100Km,Avg.C.ºC
0,Joana,Gil Navarro,1958-03-23,66722344X,OK,Spain,Valladolid,Dona,"1,469 €",3.0,Grup B,Cap de projecte,True,32108.0,25.0,10.1
1,Marc,Muñoz,1960-11-08,48840994W,OK,Spain,Alicante,H,"2,718 €",False,Grup C,Senior analyst,True,19496.0,10.4,18.7
2,Noa,Serra,1961-04-27,14308421X,OK,Spain,Alicante,D,1358 euros,4.0,Grup A,Tècnic IT,<NA>,<NA>,<NA>,16.7
3,Pol,Gil,1985-10-13,58586340F,OK,Spain,Seville,H,"1,478 €",2.0,Grup B,Data Analyst,False,<NA>,<NA>,18.3
4,David,Vila,1965-11-29,82070937P,OK,Spain,Bilbao,Dona,"1,284 €",False,Grup B,Administratiu,True,<NA>,11.8,13.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
989,Jordi,Hernández,1961-10-12,67755039Y,OK,Spain,Zaragoza,Dona,"1,074 €",1.0,Grup A,Data Analyst,True,80000.0,12.0,12.0
990,Chloé,Dubois,1958-12-02,66354268T,OK,France,Nice,H,"1,954 €",1.0,Grup B,Data Analyst,<NA>,<NA>,<NA>,11.7
991,Adrià,Vila Díaz,1991-12-31,57511543T,OK,Spain,Palma,H,"2,010 €",1.0,Grup C,Analista,False,<NA>,<NA>,-10.0
992,Sara,Serra,2002-05-28,<NA>,OK,Spain,Tarragona,D,"1,073 €",False,Grup A,Administratiu,False,<NA>,<NA>,15.8


In [175]:
display(s811.value_counts(subset=["Geo", "Country", "City"]).sort_index())

Geo  Country  City      
OK   France   Bordeaux      11
              Lille         10
              Lyon           6
              Marseille     13
              Nantes        12
              Nice          11
              Paris          8
              Toulouse      15
     Germany  Berlin         6
              Cologne        4
              Dusseldorf     5
              Frankfurt      7
              Hamburg        9
              Leipzig        5
              Munich         5
              Stuttgart      7
     Italy    Bologna       11
              Florence      18
              Genoa         12
              Naples        18
              Rodano        13
              Rome          15
              Turin         16
              Venice         5
     Norway   Bergen        10
              Oslo          14
              Stavanger      9
              Tromsø         4
              Trondheim      6
     Spain    A Coruña      46
              Alicante      44
              Barcelona     54
              Bilbao        47
              Girona        44
              Las Palmas    45
              Madrid        41
              Murcia        52
              Málaga        54
              Palma         58
              Seville       49
              Tarragona     45
              Valencia      39
              Valladolid    44
              Zaragoza      47
Name: count, dtype: int64

In [176]:
s812 = s811.drop(columns=["Geo"])

#### Process Profession

In [209]:
def fuzzy_tokenize(
  text: str | NAType,
  voc: dict[str, str]
) -> list[str]:
  def to_unicode(text: str) -> str:
    return unicodedata\
      .normalize('NFD', text)\
      .encode('ascii', 'ignore')\
      .decode('utf-8')\
      .lower()
  if pd.isna(text): return []
  return list({voc[t[0]] for t in process.extract(
    query=to_unicode(text),
    choices=list(voc),
    scorer=fuzz.WRatio,
    score_cutoff=75
  )})
def display_mappings(
  column: Series,
  mapper: Callable
):
  df = DataFrame(index=column)
  df["map"] = df.index.map(mapper)
  display(df)

In [210]:
def process_professions(df: DataFrame) -> DataFrame:
  VOC: dict[str, str] = {
    "it": "IT",
    "jr": "Junior",
    "sr": "Senior",
    "junior": "Junior",
    "senior": "Senior",
    "cap": "Lead",
    "lead": "Lead",
    "analista": "Analyst",
    "analyst": "Analyst",
    "data": "Analyst",
    "programador": "Developer",
    "programadora": "Developer",
    "dev": "Developer",
    "director": "Manager",
    "directora": "Manager",
    "responsable": "Manager",
    "administratiu": "Administrator",
    "administrativa": "Administrator",
    "admin": "Administrator",
    "tecnic": "Technician",
    "projecte": "Project",
    "vendes": "Sales"
  }
  ORD: dict[str, int] = {
    s: i for i, s in enumerate([
      "Junior",
      "Senior",
      "IT",
      "Project",
      "Sales",
      "Lead",
      "Analyst",
      "Developer",
      "Manager",
      "Administrator",
      "Technician"
    ])
  }
  def helper(s: str | NAType) -> str | NAType:
    tokens = fuzzy_tokenize(s, VOC)
    if not tokens: return pd.NA
    tokens.sort(key=lambda t: ORD[t])
    return " ".join(tokens)
  display_mappings(df["Position"].unique(), helper)
  def helper_row(row: Series) -> Series:
    row["Position"] = helper(row["Position"])
    return row
  return df.apply(helper_row, axis=1)

s813 = process_professions(s812.copy())
s813.rename(columns={"Position": "Profession"}, inplace=True)
display(s813["Profession"].value_counts(dropna=False))
display(s813)

,map
Cap de projecte,Project Lead
Senior analyst,Senior Analyst
Tècnic IT,IT Technician
Data Analyst,Analyst
Administratiu,Administrator
Responsable de vendes,Sales Manager
Analista junior,Junior Analyst
Project lead,Project Lead
Admin.,Administrator
Analyst,Analyst


,count
Profession,
Analyst,280
IT Technician,129
Project Lead,125
Sales Manager,123
Senior Analyst,122
Administrator,109
Junior Analyst,106


,Name,Surname,Birth,DNI,Country,City,M/F,Salary,Kids,Pro.G.,Profession,Car,Km/year,L/100Km,Avg.C.ºC
0,Joana,Gil Navarro,1958-03-23,66722344X,Spain,Valladolid,Dona,"1,469 €",3.0,Grup B,Project Lead,True,32108.0,25.0,10.1
1,Marc,Muñoz,1960-11-08,48840994W,Spain,Alicante,H,"2,718 €",False,Grup C,Senior Analyst,True,19496.0,10.4,18.7
2,Noa,Serra,1961-04-27,14308421X,Spain,Alicante,D,1358 euros,4.0,Grup A,IT Technician,<NA>,<NA>,<NA>,16.7
3,Pol,Gil,1985-10-13,58586340F,Spain,Seville,H,"1,478 €",2.0,Grup B,Analyst,False,<NA>,<NA>,18.3
4,David,Vila,1965-11-29,82070937P,Spain,Bilbao,Dona,"1,284 €",False,Grup B,Administrator,True,<NA>,11.8,13.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
989,Jordi,Hernández,1961-10-12,67755039Y,Spain,Zaragoza,Dona,"1,074 €",1.0,Grup A,Analyst,True,80000.0,12.0,12.0
990,Chloé,Dubois,1958-12-02,66354268T,France,Nice,H,"1,954 €",1.0,Grup B,Analyst,<NA>,<NA>,<NA>,11.7
991,Adrià,Vila Díaz,1991-12-31,57511543T,Spain,Palma,H,"2,010 €",1.0,Grup C,Analyst,False,<NA>,<NA>,-10.0
992,Sara,Serra,2002-05-28,<NA>,Spain,Tarragona,D,"1,073 €",False,Grup A,Administrator,False,<NA>,<NA>,15.8
